# Capítulo 3 — Resiliencia y testing de APIs

# Introducción

Cuando aprendemos testing por primera vez,
normalmente probamos funciones simples:

```python
sumar(2, 3)
```

Pero los sistemas reales rara vez son tan simples.

En producción, el software conversa constantemente con:

- APIs,
- microservicios,
- bases de datos,
- colas,
- redes,
- sistemas externos.

Y todos esos sistemas fallan.

---

# Idea fundamental

Los sistemas distribuidos viven en un entorno hostil.

Por eso el testing moderno no solo prueba:

```text
“¿funciona?”
```

También prueba:

```text
“¿cómo se comporta cuando todo sale mal?”
```


# Objetivos

En este capítulo aprenderás:

- cómo probar APIs,
- cómo simular fallos,
- cómo diseñar tests resilientes,
- cómo probar reintentos,
- y cómo pensar como ingeniero de sistemas reales.


In [ ]:
import requests
import time

def descargar_usuario(user_id, retries=3):
    url = f'https://jsonplaceholder.typicode.com/users/{user_id}'

    for intento in range(retries):
        try:
            response = requests.get(url)

            response.raise_for_status()

            return response.json()

        except Exception:
            time.sleep(1)

    raise Exception("No fue posible descargar el usuario")


# Primer análisis

Este código contiene muchas ideas importantes.

## Cosas interesantes

### Habla con internet

Eso implica latencia y fallos.

### Tiene reintentos

Eso implica resiliencia.

### Usa excepciones

Eso implica manejo de errores.

---

# Pregunta importante

¿Cómo probamos todos esos escenarios?

Esperar que internet falle NO es una estrategia seria de testing.


# El problema de reproducibilidad

Muchos errores reales son difíciles de reproducir.

Por ejemplo:

- timeouts aleatorios,
- fallos intermitentes,
- respuestas lentas,
- APIs inestables.

Eso vuelve el debugging extremadamente costoso.

Aquí es donde los mocks se vuelven fundamentales.


In [ ]:
from unittest.mock import patch
from requests.exceptions import Timeout

with patch("requests.get", side_effect=Timeout):
    try:
        descargar_usuario(1)

    except Exception as e:
        print(type(e))


# ¿Qué acabamos de hacer?

Simulamos un timeout sin desconectar internet.

Eso es importante porque ahora el error es:

- reproducible,
- instantáneo,
- controlado.

---

# Idea importante

Los mejores tests NO esperan que el mundo falle.

Los mejores tests CREAN fallos artificialmente.


# Simulación temporal

Muchos errores reales son temporales.

Por ejemplo:

```text
primer intento → falla
segundo intento → falla
tercer intento → éxito
```

Eso es extremadamente común en sistemas distribuidos.


In [ ]:
from unittest.mock import Mock

ok = Mock()
ok.raise_for_status.return_value = None
ok.json.return_value = {
    "id": 1,
    "name": "Ada Lovelace"
}

with patch(
    "requests.get",
    side_effect=[Timeout(), Timeout(), ok]
):
    resultado = descargar_usuario(1)

print(resultado)


# Modelo mental

```text
Intento #1 → timeout
Intento #2 → timeout
Intento #3 → éxito
```

Este tipo de comportamiento aparece constantemente en producción.

Por eso probar resiliencia es tan importante.


# Verificando comportamiento

El testing moderno no solo verifica resultados.

También verifica:

- cantidad de llamadas,
- secuencia,
- contratos,
- efectos secundarios.


In [ ]:
with patch(
    "requests.get",
    return_value=ok
) as mock_get:

    descargar_usuario(1)

    print(mock_get.call_count)
    print(mock_get.called)


# Contratos

Una API tiene contratos.

Por ejemplo:

- status codes,
- formatos JSON,
- tipos de errores,
- headers esperados.

Nuestros tests deben respetar esos contratos.

---

# Error común

Muchos estudiantes crean mocks irreales.

Eso produce tests artificiales
que no representan producción.


# Tests frágiles

Un test frágil es un test que falla por razones irrelevantes.

## Ejemplos

- depende del tiempo,
- depende del orden,
- depende de internet,
- depende de archivos reales.

---

# Problema

Los tests frágiles destruyen confianza.


# Buenas prácticas

## Mockea fronteras externas

- APIs,
- DBs,
- archivos,
- reloj.

## Simula fallos explícitamente

No esperes que ocurran naturalmente.

## Verifica comportamiento observable

No implementación interna.

## Diseña sistemas resilientes

El software real debe tolerar fallos.


# Reflexión final

El objetivo del testing resiliente no es:

```text
probar el camino feliz
```

El verdadero objetivo es:

```text
entender cómo se comporta el sistema bajo estrés
```
